[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_dnam.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_dnam.ipynb)

# Illumina Mammalian Methylation Arrays

We just need two packages for this tutorial.

In [1]:
import pandas as pd
import pyaging as pya

## Download and load example data

Let's download the publicly avaiable dataset GSE223748 with Illumina's Mammalian Methylation array. The CpG coverage of the this array (~37k) spans highly conserved CpG sequences. Let's download a subset of that data.

In [2]:
pya.data.download_example_data('GSE223748')

⏺ example data already at pyaging_data/GSE223748_subset.pkl

'pyaging_data/GSE223748_subset.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/GSE223748_subset.pkl')

In [4]:
df.head()

,cg00000165,cg00001209,cg00001364,cg00001582,cg00002920,cg00003994,cg00004555,cg00005112,cg00005271,cg00006213,...,rs7746156_II_F_C_37550,rs798149_II_F_C_37528,rs845016_II_F_C_37529,rs877309_II_F_C_37552,rs9292570_I_F_C_37499,rs9363764_II_F_C_37541,rs939290_II_F_C_37535,rs951295_I_F_C_37507,rs966367_II_F_C_37551,rs9839873_II_F_C_37532
204509080002_R01C02,0.094879,0.916154,0.890314,0.053583,0.490381,0.034852,0.159705,0.763959,0.973245,0.928975,...,0.488592,0.491361,0.480024,0.500000,0.484252,0.489448,0.505585,0.505335,0.485003,0.510081
202897220142_R04C02,0.497077,0.441263,0.915314,0.047339,0.651029,0.037774,0.082634,0.415800,0.702857,0.821715,...,0.508102,0.500299,0.507261,0.490684,0.499673,0.497256,0.564106,0.482151,0.486667,0.505236
204529320092_R01C02,0.321141,0.834158,0.881194,0.056124,0.688350,0.030225,0.086776,0.777588,0.974587,0.923934,...,0.520404,0.509568,0.507549,0.501659,0.492823,0.487243,0.516018,0.471244,0.491066,0.491759
202794570004_R02C01,0.495226,0.924121,0.915812,0.050866,0.688335,0.032344,0.113318,0.872094,0.969189,0.917076,...,0.499314,0.516132,0.487009,0.487146,0.469119,0.495125,0.548238,0.512283,0.514257,0.492520
203531420070_R05C02,0.183954,0.934332,0.924153,0.055032,0.717495,0.037108,0.098632,0.859614,0.973422,0.963446,...,0.501432,0.509412,0.485055,0.497272,0.480637,0.467502,0.494246,0.500924,0.531334,0.503709


## Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [5]:
adata = pya.pp.df_to_adata(df, imputer_strategy='knn')

This is what the `adata` object looks like:

In [6]:
adata

AnnData object with n_obs × n_vars = 100 × 37554
    var: 'percent_na'
    layers: 'X_original', None (.X)

## Predict age

### Mammalian predictors without species declaration

We can either predict one clock at once or all at the same time. Let's first start with the mammalian clocks that do not need the species Latin name for the conversion of the output into units of years.

In [7]:
pya.pred.predict_age(adata, ['Mammalian1', 'MammalianLifespan', 'MammalianFemale'])

In [8]:
adata.obs.head()

,mammalian1,mammalianlifespan,mammalianfemale
204509080002_R01C02,26.372437,93.886067,0.994351
202897220142_R04C02,1.176586,6.999176,0.991473
204529320092_R01C02,18.776438,73.335119,0.008419
202794570004_R02C01,0.890973,5.332615,0.941965
203531420070_R05C02,10.371315,68.409331,0.009133


After age prediction, the clocks are added to `adata.obs`. Moreover, the percent of missing values for each clock and other metadata are included in `adata.uns`.

In [9]:
adata

AnnData object with n_obs × n_vars = 100 × 37554
    obs: 'mammalian1', 'mammalianlifespan', 'mammalianfemale'
    var: 'percent_na'
    uns: 'mammalian1_percent_na', 'mammalian1_missing_features', 'mammalian1_metadata', 'mammalianlifespan_percent_na', 'mammalianlifespan_missing_features', 'mammalianlifespan_metadata', 'mammalianfemale_percent_na', 'mammalianfemale_missing_features', 'mammalianfemale_metadata'
    layers: 'X_original', None (.X)

### Mammalian predictors with species declaration

Mammalian2 and mammalian3 types of clocks require species declaration for the reverse transformation of the output into units of years. For the mammalian2 clocks, there are 1756 species in the dictionary with the available variables for reverse transformation; for the mammalian3, there are 1707 species. By default, Homo sapiens is the chosen species.

Let's first have a look at the species that can be used for these clocks by loading the models themselves.

In [10]:
mammalian2_model = pya.pred.load_clock('Mammalian2')
mammalian3_model = pya.pred.load_clock('Mammalian3')

We need to filter the features for the ones that are not CpG sites.

In [11]:
mammalian2_species = [feature for feature in mammalian2_model.features if feature[0:2] != 'cg']
mammalian3_species = [feature for feature in mammalian3_model.features if feature[0:2] != 'cg']
print(f"There are {len(mammalian2_species)} species Latin name features in mammalian2.")
print(f"There are {len(mammalian3_species)} species Latin name features in mammalian3.")

There are 1756 species Latin name features in mammalian2.
There are 1707 species Latin name features in mammalian3.


In [12]:
mammalian2_species[0:10]

['Anaxyrus americanus',
 'Anaxyrus boreas',
 'Anaxyrus canorus',
 'Anaxyrus cognatus',
 'Anaxyrus retiformis',
 'Anaxyrus terrestris',
 'Rhinella marina',
 'Dendrobates auratus',
 'Dendrobates leucomelas',
 'Hyla chrysoscelis']

In [13]:
mammalian3_species[0:10]

['Anaxyrus americanus',
 'Anaxyrus boreas',
 'Anaxyrus canorus',
 'Anaxyrus cognatus',
 'Rhinella marina',
 'Dendrobates auratus',
 'Dendrobates leucomelas',
 'Phyllobates vittatus',
 'Hyla chrysoscelis',
 'Hyla versicolor']

To chose a species, simply add the Latin name as a feature with value 1. In this subset version of the GSE223748 dataset, the species names are not available. Therefore, let's use the naked mole rat (Heterocephalus glaber) as our species. 

Let's first check that it is available in the clocks.

In [14]:
'Heterocephalus glaber' in  mammalian2_species

True

In [15]:
'Heterocephalus glaber' in  mammalian3_species

True

Then, let's add it as a feature to the pandas dataframe and create a new adata object.

In [16]:
df['Heterocephalus glaber'] = 1
adata = pya.pp.df_to_adata(df, imputer_strategy='knn')

Finally, let's make the predictions using the multi-tissue mammalian2 and mammalian3 clocks plus the blood-specific and skin-specific versions.

In [17]:
pya.pred.predict_age(adata, ['Mammalian2', 'MammalianSkin2', 'MammalianBlood2', 'Mammalian3', 'MammalianSkin3', 'MammalianBlood3'])

During age prediction, if the other species are not present in the input data, they will show up as missing features and the value will be automatically replaced with 0. Therefore, those missing features are not necessarily CpG sites. To double check, one can simply go to the adata.uns to check for missing features.

In [18]:
adata.uns['mammalian2_missing_features'][0:10]

['Anaxyrus americanus',
 'Anaxyrus boreas',
 'Anaxyrus canorus',
 'Anaxyrus cognatus',
 'Anaxyrus retiformis',
 'Anaxyrus terrestris',
 'Rhinella marina',
 'Dendrobates auratus',
 'Dendrobates leucomelas',
 'Hyla chrysoscelis']

Finally, let's look at the predictions.

In [19]:
adata.obs.head()

,mammalian2,mammalianskin2,mammalianblood2,mammalian3,mammalianskin3,mammalianblood3
204509080002_R01C02,14.946335,6.562780,15.321810,9.211476,4.328208,13.111774
202897220142_R04C02,17.037182,3.966518,4.293048,17.737573,0.900470,4.240264
204529320092_R01C02,12.065347,13.392643,14.483315,5.950473,7.048246,5.498531
202794570004_R02C01,15.263569,19.451386,7.148743,15.870878,14.483290,11.685419
203531420070_R05C02,6.689490,6.809801,7.602141,4.106029,2.040104,6.293626


For curiosity let's check the correlation between the clocks.

In [20]:
adata.obs.corr('pearson')

,mammalian2,mammalianskin2,mammalianblood2,mammalian3,mammalianskin3,mammalianblood3
mammalian2,1.000000,0.658934,0.609392,0.910119,0.611179,0.717135
mammalianskin2,0.658934,1.000000,0.418290,0.607472,0.900305,0.594863
mammalianblood2,0.609392,0.418290,1.000000,0.463504,0.388197,0.754871
mammalian3,0.910119,0.607472,0.463504,1.000000,0.675022,0.729099
mammalianskin3,0.611179,0.900305,0.388197,0.675022,1.000000,0.646473
mammalianblood3,0.717135,0.594863,0.754871,0.729099,0.646473,1.000000


## Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [22]:
adata.uns['mammalian2_metadata']

{'clock_name': 'mammalian2',
 'data_type': 'DNA methylation',
 'species': 'multiple species',
 'year': 2023,
 'approved_by_author': '⌛',
 'citation': 'Lu, A. T., et al. "Universal DNA methylation age across mammalian tissues." Nature Aging 3.9 (2023): 1144-1166.',
 'doi': 'https://doi.org/10.1038/s43587-023-00462-6',
 'notes': 'Universal pan-mammalian clock 2 fits relative age (age divided by species maximum lifespan) and the author inverse transformation returns species-adjusted chronological age in years.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['relative age'],
 'unit': ['years'],
 'model_type': 'elastic net regression',
 'platform': ['Horvath MammalMethylChip40'],
 'population': 'multiple mammalian species',
 'journal': 'Nature Aging',
 'last_author': 'Steve Horvath',
 'n_features': 2572,
 'citations': 390,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'postprocess': 'mammalian2',
 'reference_values': 